# Respiratory CDSS SHAP Interpretability

Bu notebook diplom uchun interpretability bosqichini tayyorlaydi. U ikki qatlamda ishlaydi:

- har doim mavjud bo'lgan NB explainability artefaktini o'qiydi
- agar `xgboost` va `shap` lokal muhitda bo'lsa, feature dataset asosida XGBoost modelini train qilib SHAP summary beradi

Muhim eslatma:
- dependency bo'lmasa notebook to'xtab qolmaydi, faqat SHAP qismi skip qilinadi
- seed dataset natijalari ilmiy yakuniy xulosa emas, texnik prototip sifatida qaraladi


In [ ]:
from __future__ import annotations

import csv
import json
from pathlib import Path
from pprint import pprint


def find_backend_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "app").exists() and (candidate / "data").exists():
            return candidate
        backend_candidate = candidate / "backend"
        if (backend_candidate / "app").exists() and (backend_candidate / "data").exists():
            return backend_candidate
    raise RuntimeError("Backend root topilmadi")


BACKEND_ROOT = find_backend_root(Path.cwd())
DATA_DIR = BACKEND_ROOT / "data"
MODEL_DIR = BACKEND_ROOT / "ml_models"

FEATURE_DATASET_PATH = DATA_DIR / "respiratory_feature_dataset.csv"
SPLIT_PATH = DATA_DIR / "respiratory_train_test_split.json"
NB_METRICS_PATH = MODEL_DIR / "respiratory_nb_metrics.json"
NB_EVALUATION_PATH = MODEL_DIR / "respiratory_nb_evaluation.json"
NB_EXPLAINABILITY_PATH = MODEL_DIR / "respiratory_nb_explainability.json"

print(f"Backend root: {BACKEND_ROOT}")
print(f"Explainability path: {NB_EXPLAINABILITY_PATH}")


In [ ]:
def load_csv_rows(path: Path) -> list[dict[str, str]]:
    with path.open("r", encoding="utf-8", newline="") as csv_file:
        return list(csv.DictReader(csv_file))


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as json_file:
        return json.load(json_file)


feature_rows = load_csv_rows(FEATURE_DATASET_PATH)
split_manifest = load_json(SPLIT_PATH)
nb_metrics = load_json(NB_METRICS_PATH)
nb_evaluation = load_json(NB_EVALUATION_PATH)
nb_explainability = load_json(NB_EXPLAINABILITY_PATH)

print(f"Feature rows loaded: {len(feature_rows)}")
print(f"Labels in explainability report: {nb_explainability['label_count']}")
print(f"Top N signals: {nb_explainability['top_n']}")


In [ ]:
print("Naive Bayes global top signals:")
for item in nb_explainability["global_top_signals"]:
    print(
        f"- {item['label']}: {item['feature']} = {item['value']} | "
        f"support={item['support_score']} | lift={item['lift_ratio']}"
    )

print("\nNB accuracy summary:")
print(f"- Holdout accuracy: {nb_metrics['metrics']['accuracy']}")
print(f"- CV accuracy: {nb_evaluation['overall_accuracy']}")


In [ ]:
for label, summary in nb_explainability["per_label"].items():
    print(f"\nLabel: {label}")
    print(f"Prior probability: {summary['prior_probability']}")
    for signal in summary["top_feature_signals"][:3]:
        print(
            f"  - {signal['feature']} = {signal['value']} | "
            f"support={signal['support_score']} | lift={signal['lift_ratio']}"
        )


## XGBoost + SHAP (optional)

Quyidagi cell'lar faqat `xgboost`, `shap` va `matplotlib` mavjud bo'lsa ishlaydi. Aks holda notebook NB explainability bilan cheklanadi.


In [ ]:
feature_names = [column for column in feature_rows[0].keys() if column not in {"sample_id", "label"}]
label_names = sorted({row['label'] for row in feature_rows})
label_to_index = {label: index for index, label in enumerate(label_names)}
row_by_id = {row['sample_id']: row for row in feature_rows}


def encode_feature_rows(rows: list[dict[str, str]], columns: list[str], value_maps=None, fit=False):
    if value_maps is None:
        value_maps = {column: {} for column in columns}
        fit = True
    encoded_rows = []

    for row in rows:
        encoded_row = []
        for column in columns:
            value = row[column]
            mapping = value_maps[column]
            if value not in mapping:
                if fit:
                    mapping[value] = len(mapping)
                else:
                    encoded_row.append(-1)
                    continue
            encoded_row.append(mapping[value])
        encoded_rows.append(encoded_row)

    return encoded_rows, value_maps


train_rows = [row_by_id[sample_id] for sample_id in split_manifest['train_ids'] if sample_id in row_by_id]
test_rows = [row_by_id[sample_id] for sample_id in split_manifest['test_ids'] if sample_id in row_by_id]

train_encoded, feature_value_maps = encode_feature_rows(train_rows, feature_names)
test_encoded, _ = encode_feature_rows(test_rows, feature_names, value_maps=feature_value_maps, fit=False)

y_train = [label_to_index[row['label']] for row in train_rows]
y_test = [label_to_index[row['label']] for row in test_rows]

print(f"Train rows: {len(train_rows)} | Test rows: {len(test_rows)}")


In [ ]:
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
    print(f"xgboost imported successfully: {xgb.__version__}")
except ImportError:
    xgb = None
    XGBOOST_AVAILABLE = False
    print("xgboost topilmadi. XGBoost + SHAP qismi skip qilinadi.")

try:
    import shap
    SHAP_AVAILABLE = True
    print(f"shap imported successfully: {shap.__version__}")
except ImportError:
    shap = None
    SHAP_AVAILABLE = False
    print("shap topilmadi. SHAP qismi skip qilinadi.")

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except ImportError:
    plt = None
    MATPLOTLIB_AVAILABLE = False
    print("matplotlib topilmadi. Plot chizish skip qilinadi.")


In [ ]:
def _to_python_list(value):
    return value.tolist() if hasattr(value, 'tolist') else value


def compute_mean_abs_shap(raw_values, feature_count: int, class_count: int):
    values = _to_python_list(raw_values)
    if not values:
        return []

    if isinstance(values, list) and values and len(values) == class_count:
        scores = []
        for feature_index in range(feature_count):
            feature_values = []
            for class_values in values:
                for row in class_values:
                    feature_values.append(abs(row[feature_index]))
            scores.append(sum(feature_values) / len(feature_values) if feature_values else 0.0)
        return scores

    if isinstance(values, list) and values and isinstance(values[0], list) and values[0] and isinstance(values[0][0], list):
        scores = []
        for feature_index in range(feature_count):
            feature_values = []
            for sample_values in values:
                for class_value in sample_values[feature_index]:
                    feature_values.append(abs(class_value))
            scores.append(sum(feature_values) / len(feature_values) if feature_values else 0.0)
        return scores

    if isinstance(values, list) and values and isinstance(values[0], list):
        scores = []
        for feature_index in range(feature_count):
            feature_values = [abs(row[feature_index]) for row in values]
            scores.append(sum(feature_values) / len(feature_values) if feature_values else 0.0)
        return scores

    return []


shap_feature_scores = None
xgb_accuracy = None

if XGBOOST_AVAILABLE and SHAP_AVAILABLE:
    target_rows = test_encoded if test_encoded else train_encoded
    target_labels = y_test if y_test else y_train

    model = xgb.XGBClassifier(
        objective="multi:softprob",
        num_class=len(label_names),
        n_estimators=80,
        max_depth=4,
        learning_rate=0.1,
        subsample=1.0,
        colsample_bytree=1.0,
        eval_metric="mlogloss",
        random_state=42,
    )
    model.fit(train_encoded, y_train)
    predictions = model.predict(target_rows)
    correct = sum(int(pred == expected) for pred, expected in zip(predictions, target_labels))
    xgb_accuracy = round(correct / len(target_labels), 3) if target_labels else 1.0

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(target_rows)
    mean_abs_scores = compute_mean_abs_shap(shap_values, len(feature_names), len(label_names))
    shap_feature_scores = sorted(
        zip(feature_names, mean_abs_scores),
        key=lambda item: item[1],
        reverse=True,
    )

    print(f"XGBoost holdout/test accuracy: {xgb_accuracy}")
    print("Top SHAP features:")
    for feature_name, score in shap_feature_scores[:10]:
        print(f"- {feature_name}: {round(float(score), 4)}")
else:
    print("SHAP analysis skipped: dependency yetishmaydi.")


In [ ]:
if shap_feature_scores and MATPLOTLIB_AVAILABLE:
    top_items = shap_feature_scores[:10]
    labels = [item[0] for item in top_items][::-1]
    values = [float(item[1]) for item in top_items][::-1]

    plt.figure(figsize=(8, 5))
    plt.barh(labels, values)
    plt.title("Top SHAP feature importance")
    plt.xlabel("Mean absolute SHAP value")
    plt.tight_layout()
    plt.show()
else:
    print("Plot skipped: SHAP natijasi yoki matplotlib mavjud emas.")


## Diplom uchun ishlatish tavsiyasi

1. Avval `respiratory_nb_explainability.json` ichidagi NB signal natijalarini metodologiya va baseline interpretatsiya bo'limiga kiriting.
2. So'ng real dataset bilan `training_xgboost.ipynb` va shu notebookni dependency bilan qayta ishga tushiring.
3. NB va XGBoost interpretatsiya natijalarini alohida jadval yoki rasm sifatida solishtiring.
4. Seed datasetdan chiqqan SHAP yoki feature signal natijalarini yakuniy klinik xulosa sifatida emas, prototip sifatida ko'rsating.
